# Assignment 3 - Disease Classification via TF-IDF and One-Hot
This notebook walks through the tasks defined in Assignment 3 using TF-IDF and One-hot encoding for medical disease classification.


## Task 1: TF-IDF Feature Extraction

In [ ]:

import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# Load data
df = pd.read_csv("../data/disease_features.csv")

# Parse stringified fields
def parse_column(col):
    return col.apply(lambda x: " ".join(ast.literal_eval(x)) if isinstance(x, str) else "")

df['Risk Factors'] = parse_column(df['Risk Factors'])
df['Symptoms'] = parse_column(df['Symptoms'])
df['Signs'] = parse_column(df['Signs'])
df['Subtypes'] = df['Subtypes'].apply(lambda x: " ".join(ast.literal_eval(x).keys()) if isinstance(x, str) else "")

# TF-IDF vectorization
tfidf_rf = TfidfVectorizer().fit_transform(df['Risk Factors'])
tfidf_symptoms = TfidfVectorizer().fit_transform(df['Symptoms'])
tfidf_signs = TfidfVectorizer().fit_transform(df['Signs'])
tfidf_subtypes = TfidfVectorizer().fit_transform(df['Subtypes'])

tfidf_combined = hstack([tfidf_rf, tfidf_symptoms, tfidf_signs, tfidf_subtypes])


## Task 2: Dimensionality Reduction (PCA & SVD)

In [ ]:

from sklearn.decomposition import PCA, TruncatedSVD
import matplotlib.pyplot as plt

pca = PCA(n_components=2).fit_transform(tfidf_combined.toarray())
svd = TruncatedSVD(n_components=2).fit_transform(tfidf_combined)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.scatter(pca[:,0], pca[:,1])
plt.title("PCA - TF-IDF")
plt.subplot(1,2,2)
plt.scatter(svd[:,0], svd[:,1])
plt.title("Truncated SVD - TF-IDF")
plt.show()


## Task 3: KNN and Logistic Regression

In [ ]:

from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

y = LabelEncoder().fit_transform(df['Disease'])
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scorers = {
    'accuracy': make_scorer(accuracy_score),
    'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
    'recall_macro': make_scorer(recall_score, average='macro', zero_division=0),
    'f1_macro': make_scorer(f1_score, average='macro', zero_division=0)
}

results = []
for k in [3, 5, 7]:
    for metric in ['euclidean', 'manhattan', 'cosine']:
        model = make_pipeline(StandardScaler(with_mean=False),
                              KNeighborsClassifier(n_neighbors=k, metric=metric))
        scores = cross_validate(model, tfidf_combined, y, cv=kf, scoring=scorers)
        results.append({'k': k, 'metric': metric, 'f1': scores['test_f1_macro'].mean()})

logreg = LogisticRegression(max_iter=1000)
log_scores = cross_validate(logreg, tfidf_combined, y, cv=kf, scoring=scorers)
print("Logistic Regression F1-score:", log_scores['test_f1_macro'].mean())


## Task 4: Critical Analysis


- **TF-IDF vs One-hot**: TF-IDF captures term relevance better, one-hot is simpler.
- **Clinical Insight**: One-hot produced tighter clusters; TF-IDF more semantically rich.
- **Limitations**: One-hot sparse, TF-IDF hard to interpret; dataset is too small for supervised learning.
